In [44]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score,balanced_accuracy_score

### Load the embeddings for the C.S.sylv sulcal region

In [81]:
ukb_embeddings = pd.read_csv('/neurospin/dico/data/deep_folding/current/models/Champollion_V0_trained_on_UKB40/SC-sylv_right/11-36-10_85_0/ukb40_random_epoch80_embeddings/full_embeddings.csv', index_col=0)
ukb_embeddings.head()

,dim1,dim2,dim3,dim4,dim5,dim6,dim7,dim8,dim9,dim10,...,dim247,dim248,dim249,dim250,dim251,dim252,dim253,dim254,dim255,dim256
ID,,,,,,,,,,,,,,,,,,,,,
sub-1000021,144.70032,-2.33259,-55.521217,-3.811033,-44.061070,41.837160,15.210489,-3.336162,-16.129618,-20.501787,...,-12.813190,-1.744127,87.089485,15.318643,-43.634010,40.511910,21.935389,5.046747,-9.521525,-53.306515
sub-1000325,189.53354,31.76580,18.947166,-8.772030,20.106237,-33.390730,17.316332,-29.190847,20.369950,-5.060660,...,17.064726,-24.591824,33.550570,47.255363,-66.175830,47.804210,-30.778520,9.909952,-13.292032,-40.064840
sub-1000458,150.20581,-1.78613,-26.557894,8.816937,-41.133255,136.199750,-2.472998,12.573650,12.101191,22.242050,...,-11.313081,5.824224,8.352239,13.173943,23.730171,-16.312475,-6.631551,6.091625,20.440187,-85.629180
sub-1000575,145.42535,-49.58505,8.499626,12.548178,-20.224136,76.969540,4.202240,-19.305752,-32.917637,25.650387,...,-14.098648,-0.068690,-18.289234,-10.241393,18.584848,-12.241503,33.521553,1.713032,8.407868,-88.648970
sub-1000606,153.36888,29.05402,-11.328059,56.158030,-35.677720,13.826549,-7.284871,21.386320,29.033829,-8.862112,...,-15.195081,14.316205,36.025223,43.307170,1.105324,-9.904149,-24.226252,-0.112310,-18.451952,2.476232


### Reduce dimension (hope to remove the noise) with a PCA

In [150]:
n_components=16

pca = PCA(n_components=n_components)
pca.fit(ukb_embeddings)
print(pca.explained_variance_ratio_)
(np.cumsum(pca.explained_variance_ratio_) < 0.99).sum()

[0.17592706 0.12306105 0.11746931 0.1103346  0.10755888 0.09084436
 0.07942355 0.07295775 0.04263993 0.02652651 0.01563623 0.01140353
 0.00637188 0.00582676 0.00307861 0.00218081]


15

In [151]:
ukb_pca_bdd = pca.transform(ukb_embeddings)

In [152]:
#scaler = StandardScaler()
#scaler.fit(ukb_embeddings)
#ukb_scl_bdd = scaler.transform(ukb_embeddings)
#ukb_scl_bdd

#### First approach: SVM trained to find the interruption

In [154]:
model = SVC(kernel='linear', probability=True,
            random_state=42,
            C=0.01, class_weight='balanced', decision_function_shape='ovr')

In [155]:
interrupted = [
'sub-1310920', #pas certain
'sub-1376904',
'sub-2863742',
'sub-3694216',
'sub-1037052',
'sub-3250551',
'sub-5401486',
'sub-1499791',
'sub-1911266',
'sub-4217758',
'sub-2693192',
'sub-1633860',
'sub-5222070',
'sub-3292254',
'sub-1613821',
'sub-2771619',
'sub-3159828',
'sub-4632483']

not_interrupted = [
'sub-3264612', 
'sub-4805237', 
'sub-1422413',
'sub-3264612', 
'sub-4805237', 
'sub-1422413',
'sub-4491384', 
'sub-2946274',
'sub-5581707',
'sub-4834994',
'sub-5437419',
'sub-5054716',
'sub-2889389',
'sub-4520944',
'sub-3009279',
'sub-1190643',
'sub-5123219',
'sub-3611294',
'sub-4788419',
]

In [156]:
X_train = ukb_embeddings.loc[interrupted + not_interrupted]
y_train = [1 for i in range(len(interrupted))] + [0 for i in range(len(not_interrupted))]
X_train_pca = pca.transform(X_train)

In [157]:
model.fit(X_train_pca, y_train)
roc_auc_score(y_train ,model.predict_proba(X_train_pca)[:,1]), balanced_accuracy_score(y_train, model.predict(X_train_pca))

(1.0, 1.0)

In [158]:
prediction = pd.DataFrame({"IID" : list(ukb_embeddings.index),
              "Pred" : model.predict_proba(ukb_pca_bdd)[:,1]})
prediction

,IID,Pred
0,sub-1000021,0.135610
1,sub-1000325,0.852249
2,sub-1000458,0.268925
3,sub-1000575,0.145422
4,sub-1000606,0.373291
...,...,...
42428,sub-6023847,0.506913
42429,sub-6024038,0.607961
42430,sub-6024150,0.384284
42431,sub-6024379,0.184552


In [159]:
(prediction.sort_values(by="Pred")[:20].IID).to_list()

['sub-4670196',
 'sub-2556049',
 'sub-3362178',
 'sub-2202452',
 'sub-3764861',
 'sub-1003454',
 'sub-4349001',
 'sub-3925158',
 'sub-1322423',
 'sub-2831240',
 'sub-3589704',
 'sub-4140019',
 'sub-3080995',
 'sub-5022437',
 'sub-1062248',
 'sub-5812035',
 'sub-4752541',
 'sub-5211300',
 'sub-4788419',
 'sub-1516808']

#### Second approach: Euclidian distance in the reduced latent space

In [91]:
from scipy.spatial import distance

In [166]:
list_dist = [distance.euclidean(pca.transform(ukb_embeddings.loc['sub-5936108'].to_numpy().reshape(1,-1)), ukb_pca_bdd[i]) for i in range(len(ukb_pca_bdd))]
df_dist = pd.DataFrame({"IID":list(ukb_embeddings.index), "Dist":list_dist})

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
/usr

In [170]:
(df_dist.sort_values(by='Dist').iloc[50:75].IID).to_list()

['sub-2590554',
 'sub-5786278',
 'sub-1630582',
 'sub-5936108',
 'sub-5766455',
 'sub-3833257',
 'sub-1444147',
 'sub-4601015',
 'sub-5987910',
 'sub-1503364',
 'sub-3179446',
 'sub-3088163',
 'sub-4262898',
 'sub-2386370',
 'sub-5691328',
 'sub-2868411',
 'sub-2944562',
 'sub-3367797',
 'sub-2044570',
 'sub-5992648',
 'sub-1440124',
 'sub-3946238',
 'sub-2660037',
 'sub-2501966',
 'sub-5792305']